In [1]:
import numpy as np
import pandas as pd

df = pd.read_parquet(r"E:\EY\EY_Project\分摊明细去敏.parquet")

df.columns.tolist()

['业务类型',
 '版本',
 '店铺标签',
 '商品类型',
 '商品名称',
 '支付方式',
 '付款类型',
 '开通类型',
 '支付时间',
 '生效时间',
 '原始过期时间',
 '过期时间',
 '生效日期',
 '原始过期日期',
 '过期日期',
 '可分摊金额',
 '当前剩余金额',
 '当前消耗金额',
 '当前消耗天数',
 '当前平均每天金额V2',
 '升级关联订单',
 '关联原订单',
 '平均每天金额',
 '总天数',
 '总月数',
 '未分摊金额',
 '`2017-02',
 '`2017-03',
 '`2017-04',
 '`2017-05',
 '`2017-06',
 '`2017-07',
 '`2017-08',
 '`2017-09',
 '`2017-10',
 '`2017-11',
 '`2017-12',
 '`2018-01',
 '`2018-02',
 '`2018-03',
 '`2018-04',
 '`2018-05',
 '`2018-06',
 '`2018-07',
 '`2018-08',
 '`2018-09',
 '`2018-10',
 '`2018-11',
 '`2018-12',
 '`2019-01',
 '`2019-02',
 '`2019-03',
 '`2019-04',
 '`2019-05',
 '`2019-06',
 '`2019-07',
 '`2019-08',
 '`2019-09',
 '`2019-10',
 '`2019-11',
 '`2019-12',
 '`2020-01',
 '`2020-02',
 '`2020-03',
 '`2020-04',
 '`2020-05',
 '`2020-06',
 '`2020-07',
 '`2020-08',
 '`2020-09',
 '`2020-10',
 '`2020-11',
 '`2020-12',
 '`2021-01',
 '`2021-02',
 '`2021-03',
 '`2021-04',
 '`2021-05',
 '`2021-06',
 '`2021-07',
 '`2021-08',
 '`2021-09',
 '`2021-10',
 '`202

In [2]:
month_cols = [c for c in df.columns if c.startswith("`20")]

len(month_cols), month_cols[:5], month_cols[-5:]

(260,
 ['`2017-02', '`2017-03', '`2017-04', '`2017-05', '`2017-06'],
 ['`2038-05', '`2038-06', '`2038-07', '`2038-08', '`2038-09'])

In [3]:
biz_cols = [
    "业务类型",
    "版本",
    "店铺标签",
    "商品类型",
    "商品名称",
    "支付方式",
    "付款类型",
    "开通类型",
]

time_cols = [
    "支付时间",
    "生效时间",
    "过期时间",
    "原始过期时间",
    "生效日期",
    "过期日期",
    "原始过期日期",
]

amount_cols = [
    "可分摊金额",
    "当前消耗金额",
    "当前剩余金额",
    "未分摊金额",
    "平均每天金额",
    "当前平均每天金额V2",
]

struct_cols = [
    "总天数",
    "总月数",
    "当前消耗天数",
    "升级关联订单",
    "关联原订单",
]

grouped_cols = (
    set(month_cols)
    | set(biz_cols)
    | set(time_cols)
    | set(amount_cols)
    | set(struct_cols)
)

unused_cols = set(df.columns) - grouped_cols

print(unused_cols)

set()


In [4]:
# 1) 识别月度列：`YYYY-MM
month_cols = [c for c in df.columns if c.startswith("`20")]
assert len(month_cols) > 0, "month_cols 为空：请检查列名前缀是否为 `20"

# 2) 月度矩阵转数值（不改原 df，只生成一个 view/copy）
M = df[month_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

# 3) 非零判定（浮点数容错）
EPS = 1e-9
NZ = (M.abs() > EPS).to_numpy()  # bool matrix (n_rows, n_months)
vals = M.to_numpy(dtype=float)
n_rows, n_months = vals.shape

print("month_cols:", n_months, "rows:", n_rows)

month_cols: 260 rows: 726932


In [5]:
# sum_month：月度摊销合计
sum_month = vals.sum(axis=1)

# nz_cnt：非零摊销月数
nz_cnt = NZ.sum(axis=1).astype(int)

# first/last non-zero index（全零行需要特殊处理）
has_nz = nz_cnt > 0

first_idx = np.full(n_rows, -1, dtype=int)
last_idx  = np.full(n_rows, -1, dtype=int)

# argmax 技巧：对 bool 求 argmax 得到首个 True（但全 False 会返回 0，所以必须用 has_nz gate）
first_idx[has_nz] = NZ[has_nz].argmax(axis=1)

# 末尾：在反转数组上找首个 True，再换算回来
rev = NZ[has_nz][:, ::-1]
last_from_end = rev.argmax(axis=1)
last_idx[has_nz] = (n_months - 1) - last_from_end

# span_month：首末非零跨度（含端点）
span_month = np.zeros(n_rows, dtype=int)
span_month[has_nz] = (last_idx[has_nz] - first_idx[has_nz] + 1)

# gap_cnt：跨度内的零月数量 = span - nz_cnt（仅对有非零者成立）
gap_cnt = np.zeros(n_rows, dtype=int)
gap_cnt[has_nz] = span_month[has_nz] - nz_cnt[has_nz]

# gap_ratio：断裂率（span 为 0 时记 0）
gap_ratio = np.zeros(n_rows, dtype=float)
gap_ratio[has_nz] = gap_cnt[has_nz] / span_month[has_nz]

df_feat = df.copy()

df_feat["sum_month"] = sum_month
df_feat["nz_cnt"] = nz_cnt
df_feat["span_month"] = span_month
df_feat["gap_cnt"] = gap_cnt
df_feat["gap_ratio"] = gap_ratio

df_feat[["sum_month","nz_cnt","span_month","gap_cnt","gap_ratio"]].describe()

,sum_month,nz_cnt,span_month,gap_cnt,gap_ratio
count,726932.000000,726932.000000,726932.000000,726932.0,726932.0
mean,3236.697357,11.636198,11.636198,0.0,0.0
std,9722.822450,7.218804,7.218804,0.0,0.0
min,-187392.960000,0.000000,0.000000,0.0,0.0
25%,0.000000,12.000000,12.000000,0.0,0.0
50%,3539.650000,13.000000,13.000000,0.0,0.0
75%,7999.000000,13.000000,13.000000,0.0,0.0
max,295000.000000,239.000000,239.000000,0.0,0.0


In [6]:
# 金额字段转数值（兜底）
amt = df_feat["可分摊金额"].astype(float)
consumed = df_feat["当前消耗金额"].astype(float)
remain = df_feat["当前剩余金额"].astype(float)

# 1) 月度合计 vs 可分摊金额：残差（应该接近 0）
df_feat["resid_month_vs_amt"] = df_feat["sum_month"] - amt

den = amt.replace(0, np.nan)
df_feat["ratio_month_to_amt"] = (df_feat["sum_month"] / den).fillna(0.0)

# 3) 可分摊金额 与（消耗+剩余）的残差（应该接近 0）
df_feat["resid_amt_vs_consumed_remain"] = amt - (consumed + remain)

# 4) 冲抵/重算类标签
df_feat["flag_negative_amt"] = (amt < 0).astype(int)
df_feat["flag_negative_sum_month"] = (df_feat["sum_month"] < 0).astype(int)

df_feat[[
    "resid_month_vs_amt",
    "ratio_month_to_amt",
    "resid_amt_vs_consumed_remain",
    "flag_negative_amt",
    "flag_negative_sum_month",
]].describe().round(3)

,resid_month_vs_amt,ratio_month_to_amt,resid_amt_vs_consumed_remain,flag_negative_amt,flag_negative_sum_month
count,726932.000,726932.000,726932.000,726932.000,726932.000
mean,-608.717,0.880,-3718.977,0.175,0.174
std,9075.703,0.325,20256.872,0.380,0.379
min,-2000000.000,0.000,-2000000.000,0.000,0.000
25%,-0.000,1.000,-21.920,0.000,0.000
50%,0.000,1.000,0.000,0.000,0.000
75%,0.000,1.000,0.000,0.000,0.000
max,872510.520,1.000,7299635.000,1.000,1.000


In [7]:
amt = df_feat["可分摊金额"].astype(float)
sum_m = df_feat["sum_month"].astype(float)

df_feat["segment_model"] = np.select(
    [
        (amt > 0) & (sum_m > 0) & (df_feat["nz_cnt"] > 0),
        (amt > 0) & (sum_m == 0),
        (amt < 0) & (sum_m < 0),
        (amt < 0) & (sum_m >= 0),
        (amt == 0),
    ],
    [
        "A1_model_core",
        "A0_zero_matrix",
        "B_offset_neg",
        "C_conflict",
        "E_amt_zero",
    ],
    default="Z_other"
)

df_feat["segment_model"].value_counts(normalize=True).round(4)

segment_model
A1_model_core     0.7066
B_offset_neg      0.1735
A0_zero_matrix    0.1156
E_amt_zero        0.0027
C_conflict        0.0016
Name: proportion, dtype: float64

In [11]:
mask_A = df_feat["segment_model"].eq("A_normal_pos") & df_feat["ratio_month_to_amt"].notna()

df_feat.loc[mask_A, ["resid_month_vs_amt", "ratio_month_to_amt", "resid_amt_vs_consumed_remain"]].describe(
    percentiles=[0.001, 0.01, 0.05, 0.1, 0.12, 0.15, 0.2, 0.25, 0.5, 0.95, 0.99, 0.999]
).round(3)

,resid_month_vs_amt,ratio_month_to_amt,resid_amt_vs_consumed_remain
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
0.1%,NaN,NaN,NaN
1%,NaN,NaN,NaN
5%,NaN,NaN,NaN
10%,NaN,NaN,NaN
12%,NaN,NaN,NaN
15%,NaN,NaN,NaN


In [13]:
df_feat["rule_amt_zero_unbalanced"] = (
    (amt == 0) & (df_feat["resid_amt_vs_consumed_remain"].abs() > 1e-6)
).astype(int)

df_feat["rule_pos_amt_zero_matrix"] = (
    (amt > 0) & (sum_m == 0)
).astype(int)

df_feat["rule_sign_conflict"] = (
    ((amt < 0) & (sum_m >= 0)) | ((amt > 0) & (sum_m < 0))
).astype(int)

mask_core = df_feat["segment_model"].eq("A1_model_core")

df_feat["rule_core_resid_large"] = (
    mask_core & (df_feat["resid_month_vs_amt"].abs() > 10)
).astype(int)

In [14]:
dt_cols = ["支付时间", "生效时间", "原始过期时间", "过期时间"]
date_cols = ["生效日期", "原始过期日期", "过期日期"]

# 1) 解析时间戳（容错：None/空字符串）
for c in dt_cols:
    df_feat[c + "_dt"] = pd.to_datetime(df_feat[c], errors="coerce")

# 2) 解析日期（如果本来是字符串，转 datetime 再取 date）
for c in date_cols:
    df_feat[c + "_dt"] = pd.to_datetime(df_feat[c], errors="coerce")

# 3) 衍生 date（仅用于对账/口径一致性）
df_feat["生效_date"] = df_feat["生效时间_dt"].dt.date
df_feat["过期_date"] = df_feat["过期时间_dt"].dt.date
df_feat["原始过期_date"] = df_feat["原始过期时间_dt"].dt.date

# 4) 日期字段（原表里已有）也做成 date
df_feat["生效日期_date"] = df_feat["生效日期_dt"].dt.date
df_feat["过期日期_date"] = df_feat["过期日期_dt"].dt.date
df_feat["原始过期日期_date"] = df_feat["原始过期日期_dt"].dt.date

# 快速缺失概览
missing_time = df_feat[[c + "_dt" for c in dt_cols]].isna().mean().sort_values(ascending=False)
missing_date = df_feat[[c + "_dt" for c in date_cols]].isna().mean().sort_values(ascending=False)
missing_time, missing_date

(原始过期时间_dt    0.117351
 生效时间_dt      0.117193
 过期时间_dt      0.117193
 支付时间_dt      0.000000
 dtype: float64,
 原始过期日期_dt    0.117351
 生效日期_dt      0.117193
 过期日期_dt      0.117193
 dtype: float64)

In [15]:
# 时间字段与日期字段对账（允许两者都缺失）
df_feat["flag_eff_date_mismatch"] = (
    df_feat["生效日期_date"].notna()
    & df_feat["生效_date"].notna()
    & (df_feat["生效日期_date"] != df_feat["生效_date"])
).astype(int)

df_feat["flag_exp_date_mismatch"] = (
    df_feat["过期日期_date"].notna()
    & df_feat["过期_date"].notna()
    & (df_feat["过期日期_date"] != df_feat["过期_date"])
).astype(int)

df_feat["flag_ori_exp_date_mismatch"] = (
    df_feat["原始过期日期_date"].notna()
    & df_feat["原始过期_date"].notna()
    & (df_feat["原始过期日期_date"] != df_feat["原始过期_date"])
).astype(int)

df_feat[[
    "flag_eff_date_mismatch",
    "flag_exp_date_mismatch",
    "flag_ori_exp_date_mismatch",
]].mean().round(4)

flag_eff_date_mismatch        0.0
flag_exp_date_mismatch        0.0
flag_ori_exp_date_mismatch    0.0
dtype: float64

In [16]:
eff = df_feat["生效时间_dt"]
exp = df_feat["过期时间_dt"]
ori = df_feat["原始过期时间_dt"]

# 服务期天数
df_feat["service_days"] = (exp - eff).dt.total_seconds() / 86400.0
df_feat["service_days"] = df_feat["service_days"].round(6)

# 服务期近似月数（用于粗对齐）
df_feat["service_months_approx"] = (df_feat["service_days"] / 30.4375).round(6)

# 改期天数（过期 - 原始过期）
df_feat["reschedule_days"] = (exp - ori).dt.total_seconds() / 86400.0
df_feat["reschedule_days"] = df_feat["reschedule_days"].round(6)

# flags
df_feat["flag_service_negative"] = (df_feat["service_days"].notna() & (df_feat["service_days"] < 0)).astype(int)
df_feat["flag_reschedule"] = (df_feat["reschedule_days"].abs().fillna(0) > 0).astype(int)

df_feat[["service_days", "service_months_approx", "reschedule_days"]].describe().round(3)

,service_days,service_months_approx,reschedule_days
count,641741.000,641741.000,641626.000
mean,374.231,12.295,11.873
std,186.480,6.127,90.773
min,-64.635,-2.124,-1558.746
25%,365.000,11.992,0.000
50%,365.000,11.992,0.000
75%,366.000,12.025,0.000
max,7504.280,246.547,3650.000


In [17]:
# 基础 datetime
pay = df_feat["支付时间_dt"]
eff = df_feat["生效时间_dt"]
exp = df_feat["过期时间_dt"]

# A) 支付 -> 生效 延迟（天）
df_feat["pay_to_eff_days"] = (eff - pay).dt.total_seconds() / 86400.0

# flags：回溯生效（生效早于支付）
df_feat["flag_pay_after_eff"] = (df_feat["pay_to_eff_days"].notna() & (df_feat["pay_to_eff_days"] < 0)).astype(int)

# B) 对账：服务期 vs 系统总天数/总月数（仅在不缺失时）
df_feat["delta_service_days_vs_total_days"] = np.nan
mask_days = df_feat["service_days"].notna() & df_feat["总天数"].notna()
df_feat.loc[mask_days, "delta_service_days_vs_total_days"] = df_feat.loc[mask_days, "service_days"] - df_feat.loc[mask_days, "总天数"].astype(float)

df_feat["delta_service_months_vs_total_months"] = np.nan
mask_months = df_feat["service_months_approx"].notna() & df_feat["总月数"].notna()
df_feat.loc[mask_months, "delta_service_months_vs_total_months"] = df_feat.loc[mask_months, "service_months_approx"] - df_feat.loc[mask_months, "总月数"].astype(float)

# C) 时间异常 flags（阈值先用保守值，后续可再调）
df_feat["flag_service_too_long_5y"] = (df_feat["service_days"].notna() & (df_feat["service_days"] > 365*5)).astype(int)
df_feat["flag_reschedule_large_90d"] = (df_feat["reschedule_days"].notna() & (df_feat["reschedule_days"].abs() > 90)).astype(int)

# 快速看看占比（导出前自检）
df_feat[[
    "flag_service_negative",
    "flag_service_too_long_5y",
    "flag_reschedule_large_90d",
    "flag_pay_after_eff"
]].mean().round(4)

flag_service_negative        0.0000
flag_service_too_long_5y     0.0012
flag_reschedule_large_90d    0.0540
flag_pay_after_eff           0.0006
dtype: float64

In [18]:
df_feat["abs_resid_month_vs_amt"] = df_feat["resid_month_vs_amt"].abs()
df_feat["abs_resid_amt_vs_consumed_remain"] = df_feat["resid_amt_vs_consumed_remain"].abs()

mask_time_ok = df_feat["service_months_approx"].notna()
amort_months = df_feat["span_month"].astype(float)

df_feat["delta_amort_vs_service_months"] = np.nan
df_feat.loc[mask_time_ok, "delta_amort_vs_service_months"] = amort_months[mask_time_ok] - df_feat.loc[mask_time_ok, "service_months_approx"]
df_feat["abs_delta_amort_vs_service_months"] = df_feat["delta_amort_vs_service_months"].abs()
df_feat["flag_amort_service_mismatch_2m"] = (df_feat["abs_delta_amort_vs_service_months"].fillna(0) > 2).astype(int)

In [21]:
fact_cols = (
    biz_cols
    + ["支付时间","生效时间","原始过期时间","过期时间","生效日期","原始过期日期","过期日期"]
    + ["支付时间_dt","生效时间_dt","原始过期时间_dt","过期时间_dt"]  # 可选：保留计算字段
    + [
        "可分摊金额","当前消耗金额","当前剩余金额","未分摊金额",
        "总天数","总月数","平均每天金额","当前平均每天金额V2","当前消耗天数",
        "sum_month","nz_cnt","span_month","gap_cnt","gap_ratio",
        "resid_month_vs_amt","ratio_month_to_amt","resid_amt_vs_consumed_remain",
        "segment_model",
        # 时间特征
        "service_days","service_months_approx","reschedule_days","pay_to_eff_days",
        "delta_service_days_vs_total_days","delta_service_months_vs_total_months",
        # flags
        "flag_negative_amt","flag_negative_sum_month",
        "flag_service_negative","flag_service_too_long_5y","flag_reschedule_large_90d","flag_pay_after_eff",
        "abs_resid_month_vs_amt", "abs_resid_amt_vs_consumed_remain", "delta_amort_vs_service_months", 
        "abs_delta_amort_vs_service_months", "flag_amort_service_mismatch_2m"
    ]
)

fact_object = df_feat[fact_cols].copy()

In [22]:
print(fact_object.shape)

(726932, 54)


In [23]:
fact_object.columns.tolist()

['业务类型',
 '版本',
 '店铺标签',
 '商品类型',
 '商品名称',
 '支付方式',
 '付款类型',
 '开通类型',
 '支付时间',
 '生效时间',
 '原始过期时间',
 '过期时间',
 '生效日期',
 '原始过期日期',
 '过期日期',
 '支付时间_dt',
 '生效时间_dt',
 '原始过期时间_dt',
 '过期时间_dt',
 '可分摊金额',
 '当前消耗金额',
 '当前剩余金额',
 '未分摊金额',
 '总天数',
 '总月数',
 '平均每天金额',
 '当前平均每天金额V2',
 '当前消耗天数',
 'sum_month',
 'nz_cnt',
 'span_month',
 'gap_cnt',
 'gap_ratio',
 'resid_month_vs_amt',
 'ratio_month_to_amt',
 'resid_amt_vs_consumed_remain',
 'segment_model',
 'service_days',
 'service_months_approx',
 'reschedule_days',
 'pay_to_eff_days',
 'delta_service_days_vs_total_days',
 'delta_service_months_vs_total_months',
 'flag_negative_amt',
 'flag_negative_sum_month',
 'flag_service_negative',
 'flag_service_too_long_5y',
 'flag_reschedule_large_90d',
 'flag_pay_after_eff',
 'abs_resid_month_vs_amt',
 'abs_resid_amt_vs_consumed_remain',
 'delta_amort_vs_service_months',
 'abs_delta_amort_vs_service_months',
 'flag_amort_service_mismatch_2m']

In [ ]:
fact_object.describe().round(2)

In [ ]:
fact_object.head()

In [25]:
fact_object = fact_object.copy()

In [26]:
import pandas as pd

# ========= 你只需要保证这里的变量名对应你的 notebook =========
# df: 含 `YYYY-MM` 月份列的原表（你之前叫 org / df_org 就替换）
# fact_object: 你要作为“事实宽表”的输出df（目前缺 id 和月份列）

assert "fact_object" in globals(), "找不到 fact_object 变量"
assert "df" in globals(), "找不到 df 变量（含月份列的原表）"

# 1) 抽取并排序月份列
month_cols = [c for c in df.columns if c.startswith("`20")]
assert len(month_cols) > 0, "df 里没找到月份列（应形如 `2017-02）"
month_cols = sorted(month_cols, key=lambda x: x.strip("`"))

# 2) 行数一致性校验（按你说的“顺序一致”前提）
assert len(fact_object) == len(df), f"行数不一致：fact_object={len(fact_object)} vs df={len(df)}，不能按顺序 join"

# 3) 生成顺序 object_id（000001 开始）
#   - 默认生成 6 位，不够会自动扩位（比如 700000 -> 700000）
n = len(fact_object)
width = max(6, len(str(n)))
fact_object["object_id"] = [str(i).zfill(width) for i in range(1, n + 1)]

# 4) 把月份列按顺序 join 回 fact_object
#   - 直接赋值更快更省内存；也避免 join 的索引对齐坑
fact_object[month_cols] = df[month_cols].values

# 5) 把 object_id 放到第一列（可选，但建议）
cols = ["object_id"] + [c for c in fact_object.columns if c != "object_id"]
fact_object = fact_object[cols]

print("fact_object 已补齐 object_id + 月份列")
print("fact_object shape:", fact_object.shape)
print("month_cols:", len(month_cols), "from", month_cols[0], "to", month_cols[-1])
print("object_id preview:", fact_object["object_id"].head().tolist())

fact_object 已补齐 object_id + 月份列
fact_object shape: (726932, 315)
month_cols: 260 from `2017-02 to `2038-09
object_id preview: ['000001', '000002', '000003', '000004', '000005']


In [27]:
fact_object.columns.tolist()

['object_id',
 '业务类型',
 '版本',
 '店铺标签',
 '商品类型',
 '商品名称',
 '支付方式',
 '付款类型',
 '开通类型',
 '支付时间',
 '生效时间',
 '原始过期时间',
 '过期时间',
 '生效日期',
 '原始过期日期',
 '过期日期',
 '支付时间_dt',
 '生效时间_dt',
 '原始过期时间_dt',
 '过期时间_dt',
 '可分摊金额',
 '当前消耗金额',
 '当前剩余金额',
 '未分摊金额',
 '总天数',
 '总月数',
 '平均每天金额',
 '当前平均每天金额V2',
 '当前消耗天数',
 'sum_month',
 'nz_cnt',
 'span_month',
 'gap_cnt',
 'gap_ratio',
 'resid_month_vs_amt',
 'ratio_month_to_amt',
 'resid_amt_vs_consumed_remain',
 'segment_model',
 'service_days',
 'service_months_approx',
 'reschedule_days',
 'pay_to_eff_days',
 'delta_service_days_vs_total_days',
 'delta_service_months_vs_total_months',
 'flag_negative_amt',
 'flag_negative_sum_month',
 'flag_service_negative',
 'flag_service_too_long_5y',
 'flag_reschedule_large_90d',
 'flag_pay_after_eff',
 'abs_resid_month_vs_amt',
 'abs_resid_amt_vs_consumed_remain',
 'delta_amort_vs_service_months',
 'abs_delta_amort_vs_service_months',
 'flag_amort_service_mismatch_2m',
 '`2017-02',
 '`2017-03',
 '`2017-04',
 '`2017-05',

In [28]:
import re

# 1) 固定列映射
rename_map = {
    "object_id": "obj_id",
    "业务类型": "biz_type",
    "版本": "prod_ver",
    "店铺标签": "shop_tag",
    "商品类型": "item_type",
    "商品名称": "item_name",
    "支付方式": "pay_method",
    "付款类型": "pay_type",
    "开通类型": "activate_type",

    "支付时间": "pay_ts",
    "生效时间": "eff_ts",
    "原始过期时间": "exp0_ts",
    "过期时间": "exp_ts",

    "生效日期": "eff_date",
    "原始过期日期": "exp0_date",
    "过期日期": "exp_date",

    "支付时间_dt": "pay_dt",
    "生效时间_dt": "eff_dt",
    "原始过期时间_dt": "exp0_dt",
    "过期时间_dt": "exp_dt",

    "可分摊金额": "amt_alloc",
    "当前消耗金额": "amt_used",
    "当前剩余金额": "amt_rem",
    "未分摊金额": "amt_unalloc",

    "总天数": "days_total",
    "总月数": "mos_total",
    "平均每天金额": "amt_day_avg",
    "当前平均每天金额V2": "amt_day_avg_v2",
    "当前消耗天数": "days_used",

    "sum_month": "m_sum",
    "nz_cnt": "m_nz_cnt",
    "span_month": "m_span",
    "gap_cnt": "m_gap_cnt",
    "gap_ratio": "m_gap_rt",

    "resid_month_vs_amt": "res_m_sum_vs_amt",
    "ratio_month_to_amt": "rt_m_sum_to_amt",
    "resid_amt_vs_consumed_remain": "res_amt_used_rem",

    "segment_model": "seg_grp",

    "service_days": "svc_days",
    "service_months_approx": "svc_mos_apx",
    "reschedule_days": "resched_days",
    "pay_to_eff_days": "pay2eff_days",
    "delta_service_days_vs_total_days": "d_svc_days_vs_total",
    "delta_service_months_vs_total_months": "d_svc_mos_vs_total",
    "delta_amort_vs_service_months": "d_amort_mos_vs_svc",
    "abs_delta_amort_vs_service_months": "abs_d_amort_mos_vs_svc",

    "flag_negative_amt": "f_amt_neg",
    "flag_negative_sum_month": "f_m_sum_neg",
    "flag_service_negative": "f_svc_neg",
    "flag_service_too_long_5y": "f_svc_gt_5y",
    "flag_reschedule_large_90d": "f_resched_gt_90d",
    "flag_pay_after_eff": "f_pay_after_eff",
    "flag_amort_service_mismatch_2m": "f_amort_svc_mis_2m",

    "abs_resid_month_vs_amt": "abs_res_m_sum_vs_amt",
    "abs_resid_amt_vs_consumed_remain": "abs_res_amt_used_rem",
}

# 2) 月份列映射：`YYYY-MM` -> m_YYYY_MM
month_pat = re.compile(r"^`(\d{4})-(\d{2})$")
for c in list(fact_object.columns):
    m = month_pat.match(c)
    if m:
        y, mm = m.group(1), m.group(2)
        rename_map[c] = f"m_{y}_{mm}"

# 3) 执行 rename
fact_object = fact_object.rename(columns=rename_map)

print("OK: renamed. cols preview:", list(fact_object.columns)[:25])

OK: renamed. cols preview: ['obj_id', 'biz_type', 'prod_ver', 'shop_tag', 'item_type', 'item_name', 'pay_method', 'pay_type', 'activate_type', 'pay_ts', 'eff_ts', 'exp0_ts', 'exp_ts', 'eff_date', 'exp0_date', 'exp_date', 'pay_dt', 'eff_dt', 'exp0_dt', 'exp_dt', 'amt_alloc', 'amt_used', 'amt_rem', 'amt_unalloc', 'days_total']


In [29]:
fact_object.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726932 entries, 0 to 726931
Columns: 315 entries, obj_id to m_2038_09
dtypes: datetime64[ns](4), float64(281), int32(10), int64(3), object(17)
memory usage: 1.7+ GB


In [30]:
fact_object.to_parquet(r"E:\EY\EY_Project\fact_object.parquet", index=False, engine="pyarrow", compression="zstd")